# Earthquakes Analysis with Elasticsearch

Dans ce notebook, nous allons interroger les données de tremblements de terre et de blasts
ingérées dans Elasticsearch par le conteneur `ingest-init`.

Indices utilisés :
- `ncedc-earthquakes-earthquake`
- `ncedc-earthquakes-blast`

Assurez-vous que :
- `docker compose up -d` a été lancé
- le job d'ingestion a terminé (voir logs du conteneur `ingest-init`)

In [ ]:
!pip install "elasticsearch>=8.0.0,<10.0.0"

In [ ]:
from elasticsearch import Elasticsearch

ES_HOST = "http://elasticsearch:9200"
ES_USER = "elastic"
ES_PASSWORD = "111"  # ou récupérer via variable d'env si tu préfères

es = Elasticsearch(
    ES_HOST,
    basic_auth=(ES_USER, ES_PASSWORD),
)

if es.ping():
    print("Connected to Elasticsearch!")
else:
    raise RuntimeError("Could not connect to Elasticsearch")


In [ ]:
indices = ["ncedc-earthquakes-earthquake", "ncedc-earthquakes-blast"]

for index in indices:
    if not es.indices.exists(index=index):
        es.indices.create(index=index)
        print(f"Created empty index: {index}")
    else:
        print(f"Index already exists: {index}")

In [ ]:
for index in indices:
    if es.indices.exists(index=index):
        count = es.count(index=index)["count"]
        print(f"{index}: {count} documents")
    else:
        print(f"{index} does NOT exist")

In [ ]:
import os
print(os.getcwd())
print(os.listdir())


Si il fonctionne pas il faut lancer la commande 

docker cp ncedc-earthquakes-dataset jupyter:/home/jovyan/work/


In [ ]:
from pathlib import Path
import os

print("CWD:", os.getcwd())
print("Contenu de work:", os.listdir("/home/jovyan/work"))

SOURCE_DIR = Path("/home/jovyan/work/ncedc-earthquakes-dataset")
print("Exists:", SOURCE_DIR.exists())
print("Files:", list(SOURCE_DIR.glob("*")))

In [ ]:
import json
from pathlib import Path
import requests

ES_URL = "http://elasticsearch:9200"
AUTH = ("elastic", "111")
SOURCE_DIR = Path("/home/jovyan/work/ncedc-earthquakes-dataset")  # adapte le chemin au volume monté

def bulk_ingest(file_path: Path, index_name: str, event_type: str, limit: int | None = 5000):
    buffer = []
    count = 0

    def flush():
        nonlocal buffer
        if not buffer:
            return
        payload = "\n".join(buffer) + "\n"
        r = requests.post(
            f"{ES_URL}/_bulk?pipeline=ncedc-earthquakes",
            data=payload.encode("utf-8"),
            headers={"Content-Type": "application/x-ndjson"},
            auth=AUTH,
            timeout=60,
        )
        r.raise_for_status()
        resp = r.json()
        if resp.get("errors"):
            raise RuntimeError("Bulk ingest error")
        buffer = []

    with file_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            buffer.append(json.dumps({"index": {"_index": index_name}}))
            buffer.append(json.dumps({"message": line, "type": event_type}))
            count += 1
            if count % 1000 == 0:
                flush()
            if limit is not None and count >= limit:
                break

    flush()
    print(f"Ingested {count} lines into {index_name}")

bulk_ingest(SOURCE_DIR / "earthquakes.txt", "ncedc-earthquakes-earthquake", "earthquake")
bulk_ingest(SOURCE_DIR / "blasts.txt", "ncedc-earthquakes-blast", "blast")

In [ ]:
resp = es.search(
    index="ncedc-earthquakes-earthquake",
    size=5,
    query={"match_all": {}},
)
for hit in resp["hits"]["hits"]:
    print(hit["_source"])


In [ ]:
body = {
    "size": 0,
    "aggs": {
        "mag_hist": {
            "histogram": {
                "field": "mag",
                "interval": 0.5,
                "min_doc_count": 1,
            }
        }
    },
}

resp_eq = es.search(index="ncedc-earthquakes-earthquake", body=body)
buckets = resp_eq["aggregations"]["mag_hist"]["buckets"]
buckets[:5]

## Distribution des magnitudes

On commence par un histogramme de la magnitude (`mag`) pour les tremblements de terre.
Chaque bucket représente un intervalle de 0.5 de magnitude.

In [ ]:
def mag_hist(index_name: str):
    body = {
        "size": 0,
        "aggs": {
            "mag_hist": {
                "histogram": {
                    "field": "mag",
                    "interval": 0.5,
                    "min_doc_count": 1,
                }
            }
        },
    }
    resp = es.search(index=index_name, body=body)
    return resp["aggregations"]["mag_hist"]["buckets"]

eq_buckets = mag_hist("ncedc-earthquakes-earthquake")
blast_buckets = mag_hist("ncedc-earthquakes-blast")


In [ ]:
body = {
    "size": 0,
    "aggs": {
        "per_month": {
            "date_histogram": {
                "field": "@timestamp",
                "calendar_interval": "1M",
                "min_doc_count": 1,
            }
        }
    },
}

resp_eq_time = es.search(index="ncedc-earthquakes-earthquake", body=body)
time_buckets = resp_eq_time["aggregations"]["per_month"]["buckets"]
time_buckets[:5]


## Évolution du nombre de tremblements de terre dans le temps

On regroupe les événements par mois en utilisant un `date_histogram` sur `@timestamp`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def buckets_to_df(buckets, label):
    return pd.DataFrame(
        {
            "mag": [b["key"] for b in buckets],
            "count": [b["doc_count"] for b in buckets],
            "type": label,
        }
    )

df_eq = buckets_to_df(eq_buckets, "earthquake")
df_blast = buckets_to_df(blast_buckets, "blast")

df_mag = pd.concat([df_eq, df_blast], ignore_index=True)
df_mag.head()


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=df_mag,
    x="mag",
    y="count",
    hue="type",
)
plt.title("Distribution des magnitudes : earthquakes vs blasts")
plt.xlabel("Magnitude (bins de 0.5)")
plt.ylabel("Nombre d'événements")
plt.tight_layout()
plt.show()


In [ ]:
df_time = pd.DataFrame(
    {
        "date": [pd.to_datetime(b["key_as_string"]) for b in time_buckets],
        "count": [b["doc_count"] for b in time_buckets],
    }
)

plt.figure(figsize=(12, 4))
plt.plot(df_time["date"], df_time["count"])
plt.title("Nombre de tremblements de terre par mois")
plt.xlabel("Date")
plt.ylabel("Nombre d'événements")
plt.tight_layout()
plt.show()


## Aller plus loin dans Kibana

Pour créer des visualisations plus avancées (cartes, dashboards), suivez le guide
**Earthquakes Ingestion Guide** (`data_visualization.md`) dans le dépôt.

Les mêmes index sont utilisés :
- `ncedc-earthquakes-earthquake`
- `ncedc-earthquakes-blast`
